## TH 5.1

Dựng tam giác Pascal $10$ hàng trong ma trận số nguyên $10\times10$ bằng *slicing* (mỗi hàng tính từ hàng trước bằng một phép cộng vector, không dùng vòng lặp lồng): $P_{i,j}=P_{i-1,j}+P_{i-1,j-1}$, $P_{i,0}=1$. In hàng $4$ và hàng $9$; kiểm tra $\binom94=126$ và tổng hàng $9$ bằng $2^9$.

In [1]:
import numpy as np
from math import comb

n = 10
P = np.zeros((n, n), dtype=int)
P[:, 0] = 1                                 
for i in range(1, n):                        
    P[i, 1:] = P[i-1, 1:] + P[i-1, :-1]      

print("Hàng 4:", P[4, :5])                  
print("Hàng 9:", P[9])                      

print(P[9, 4] == 126, P[9, 4] == comb(9, 4))   
print(P[9].sum() == 2**9)                      

Hàng 4: [1 4 6 4 1]
Hàng 9: [  1   9  36  84 126 126  84  36   9   1]
True True
True


## TH 5.5

Ma trận $N=\begin{pmatrix}0&1&2
0&0&3
0&0&0\end{pmatrix}$ thoả $N^3=0$ (luỹ linh). Kiểm tra bằng NumPy: (i) $N^3=0$; (ii) $(I-N)^{-1}=I+N+N^2$; (iii) $(I+N)^{-1}=I-N+N^2$. Giải thích đẳng thức (ii) bằng nhân hai vế với $(I-N)$.

In [2]:
N = np.array([[0, 1, 2],
              [0, 0, 3],
              [0, 0, 0]])
I = np.eye(3)

print(np.allclose(np.linalg.matrix_power(N, 3), 0))                 
print(np.allclose(np.linalg.inv(I - N), I + N + N @ N))             
print(np.allclose(np.linalg.inv(I + N), I - N + N @ N))             

True
True
True


## TH 5.6

Cho $A=\begin{pmatrix}2&1&-1&0
1&3&2&1
0&-1&4&2
3&0&1&5\end{pmatrix}$ và véc tơ nghiệm biết trước $\mathbf x^*=(1,-1,2,3)^\top$. Đặt $\mathbf b=A\mathbf x^*$, giải lại hệ bằng `np.linalg.solve`, kiểm chứng bằng `np.allclose`. Sau đó thay hàng $4$ của $A$ bằng tổng hàng $1$ và hàng $2$ và ghi lại lỗi của `solve`.

In [3]:
A = np.array([[2,  1, -1, 0],
              [1,  3,  2, 1],
              [0, -1,  4, 2],
              [3,  0,  1, 5]], dtype=float)
x_star = np.array([1, -1, 2, 3], dtype=float)
b = A @ x_star                                

x = np.linalg.solve(A, b)
print(x, np.allclose(x, x_star))              
print(np.allclose(A @ x, b))                  

A2 = A.copy()
A2[3] = A[0] + A[1]                           

try:
    np.linalg.solve(A2, b)
except np.linalg.LinAlgError as e:
    print("Lỗi:", e)                          

[ 1. -1.  2.  3.] True
True


## TH 5.7

Đa thức bậc $3$ đi qua bốn điểm $(0,1)$, $(1,0)$, $(2,5)$, $(3,22)$ là duy nhất. Tìm các hệ số $c_0,…,c_3$ của $p(x)=c_0+c_1x+c_2x^2+c_3x^3$ bằng cách giải hệ với *ma trận Vandermonde* (`np.vander(x, increasing=True)`), kiểm tra bằng `allclose`, rồi tính $p(1{,}5)$.

In [4]:
import numpy as np

x = np.array([0, 1, 2, 3], dtype=float)
y = np.array([1, 0, 5, 22], dtype=float)

V = np.vander(x, increasing=True)     
c = np.linalg.solve(V, y)             
print("Hệ số:", c)                    

print(np.allclose(V @ c, y))          

p = lambda t: c[0] + c[1]*t + c[2]*t**2 + c[3]*t**3
print("p(1.5) =", p(1.5))            

Hệ số: [ 1.00000000e+00 -2.00000000e+00 -1.33226763e-15  1.00000000e+00]
True
p(1.5) = 1.3749999999999982


## TH 5.8

Với ma trận ngẫu nhiên $500\times500$ (`default_rng(0)`) và véc tơ $\mathbf b$ ngẫu nhiên, giải $A\mathbf x=\mathbf b$ bằng `solve` và bằng `inv(A) @ b`. So sánh *phần dư* $\|A\mathbf x-\mathbf b\|$ và thời gian của hai cách; kiểm tra bằng `assert` cả hai nghiệm đều đạt phần dư rất nhỏ và giống nhau.

In [6]:
import numpy as np
import time

rng = np.random.default_rng(0)
n = 500
A = rng.standard_normal((n, n))
b = rng.standard_normal(n)

t0 = time.perf_counter()
x1 = np.linalg.solve(A, b)
t_solve = time.perf_counter() - t0

t0 = time.perf_counter()
x2 = np.linalg.inv(A) @ b
t_inv = time.perf_counter() - t0

r1 = np.linalg.norm(A @ x1 - b)
r2 = np.linalg.norm(A @ x2 - b)

print(f"solve : residual = {r1:.2e}, time = {t_solve*1e3:.2f} ms")
print(f"inv@b : residual = {r2:.2e}, time = {t_inv*1e3:.2f} ms")

solve : residual = 1.58e-11, time = 6.62 ms
inv@b : residual = 1.76e-11, time = 18.17 ms
